# Notebook_C (dùng chung, phần dữ liệu, Sinh viên A): thu thập, so sánh nguồn, làm sạch, EDA và SQL

**Dùng chung cho mọi đề tài của môn.** Chỉ cần điền khối CONFIG bên dưới (tên đề tài, nguồn, tên cột, tần suất) rồi chạy tuần tự. Mọi ô code có chú thích tiếng Anh; phần giải thích, lý do và cách đọc kết quả bằng tiếng Việt nằm ngay trên mỗi ô.

**Sản phẩm bàn giao cho Sinh viên B**: `data/processed/feat.parquet`, `data/processed/manifest.json`, thư mục `report/` (bảng RQ1, hình, nhật ký làm sạch, AI Audit Log).

**Ba câu hỏi nghiên cứu chung (áp dụng cho mọi đề, mỗi nhóm thêm 1-2 câu riêng theo đề):**

* **RQ1 (dữ liệu và mô tả, sâu hơn)**: Hai nguồn dữ liệu (chính và phụ) có nhất quán với nhau không, biến mục tiêu biến động theo mùa, theo đơn vị và theo biến phụ như thế nào, và phần nào của biến động được giải thích bằng lịch so với bằng nguồn phụ?
* **RQ2 (mô hình và tổng quát hoá)**: Trong sáu họ mô hình (mốc naive, ridge, random forest, LightGBM toàn cục, tuyến tính dài hạn, mô hình nền zero-shot), mô hình nào tốt nhất theo tầm dự báo, và kết quả có giữ được khi lùi gốc dự báo theo thời gian và khi gặp đơn vị chưa thấy không?
* **RQ3 (tin cậy và vận hành)**: Khoảng tin cậy có đúng bao phủ không, cảnh báo sự kiện (mục tiêu vượt phân vị cao) đạt recall và precision bao nhiêu, sai số thay đổi thế nào khi chế độ đổi, và nguồn phụ đóng góp bao nhiêu (ablation)?

Notebook_C trả lời RQ1 và chuẩn bị dữ liệu cho RQ2, RQ3 (Notebook_D).

In [16]:
TOPIC = "Dự báo tải sạc EV theo giờ"
GROUP = "Nhom 11"

PRIMARY_SOURCE = {
    "name": "ACN-Data EV charging",
    "url": "https://ev.caltech.edu/dataset",
    "license": "research/public dataset",
    "path": "data/raw/acn_hourly.csv"
}

SECOND_SOURCE = {
    "name": "NOAA Pasadena hourly weather",
    "url": "https://www.ncei.noaa.gov/",
    "license": "public data",
    "path": "data/raw/pasadena_hourly.csv"
}

UNIT_COL = "site"
TIME_COL = "ts"
TARGET_COL = "kwh"

EXOG_COLS = [
    "temperature",
    "humidity",
    "wind_speed",
    "precipitation"
]

FREQ = "h"
HORIZONS = [1, 24]
SEASON = 24
TEST_START = "2020-01-01"
EVENT_QUANTILE = 0.9

print("Topic:", TOPIC)
print("Group:", GROUP)

Topic: Dự báo tải sạc EV theo giờ
Group: Nhom 11


In [77]:
import pandas as pd
import numpy as np

wx = pd.read_csv(
    "data/raw/noaa/LCD_USW00023152_2018_2021.csv",
    low_memory=False
)

# Time
wx["ts"] = (
    pd.to_datetime(wx["DATE"], errors="coerce")
    .dt.floor("h")
    + pd.Timedelta(hours=8)
)

# Convert weather columns to numeric
def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0],
        errors="coerce"
    )

wx["temperature"] = to_num(wx["HourlyDryBulbTemperature"])
wx["humidity"] = to_num(wx["HourlyRelativeHumidity"])
wx["wind_speed"] = to_num(wx["HourlyWindSpeed"])

# NOAA may use T = trace precipitation
precip = wx["HourlyPrecipitation"].astype(str).str.strip()
precip = precip.replace({"T": "0", "nan": np.nan})
wx["precipitation"] = to_num(precip)

weather = (
    wx[["ts", "temperature", "humidity", "wind_speed", "precipitation"]]
    .dropna(subset=["ts"])
    .groupby("ts", as_index=False)
    .mean()
    .sort_values("ts")
)

weather.to_csv(
    "data/raw/pasadena_hourly.csv",
    index=False
)

print("Shape:", weather.shape)
print("Time:", weather["ts"].min(), "->", weather["ts"].max())
print("Missing:")
print(weather.isna().mean().round(3))

display(weather.head())

Shape: (35050, 5)
Time: 2018-01-01 08:00:00 -> 2022-01-01 07:00:00
Missing:
ts               0.000
temperature      0.000
humidity         0.000
wind_speed       0.002
precipitation    0.009
dtype: float64


,ts,temperature,humidity,wind_speed,precipitation
0,2018-01-01 08:00:00,9.4,66.0,2.6,0.0
1,2018-01-01 09:00:00,9.4,69.0,0.0,0.0
2,2018-01-01 10:00:00,8.9,68.0,0.0,0.0
3,2018-01-01 11:00:00,8.9,71.0,0.0,0.0
4,2018-01-01 12:00:00,7.8,73.0,0.0,0.0


In [17]:
# AI Audit Log helper: every prompt that changed your work is one row (2 minutes per entry, 3-5 per week)
import pandas as pd, os, datetime as dt
os.makedirs("report", exist_ok=True)
AUDIT_PATH = "report/ai_audit_log.csv"
def audit(step, prompt, tool, ai_output_summary, verified_how, decision, hallucination=False):
    """Append one entry. decision: what you kept / changed / rejected. hallucination=True if the AI answer was wrong and you caught it."""
    row = {"date": dt.date.today().isoformat(), "group": GROUP, "step": step, "prompt": prompt[:500], "tool": tool,
           "ai_output": ai_output_summary[:500], "verified_how": verified_how[:300], "decision": decision[:300], "hallucination": int(hallucination)}
    df = pd.DataFrame([row])
    df.to_csv(AUDIT_PATH, mode="a", header=not os.path.exists(AUDIT_PATH), index=False)
    print("audit entry saved:", step)
# example (delete after reading): audit("Step 2", "Given columns ... how to treat gaps longer than 3 steps?", "Claude", "suggested interpolate(limit=3) then drop", "checked share of gaps > 3 in Q4 below", "kept limit=3, dropped 1.2% rows")

## Hướng dẫn hỏi AI (điền đề tài của nhóm vào chỗ trống)

Quy tắc: hỏi **cụ thể** (kèm tên cột, kích thước, thông báo lỗi), yêu cầu AI **giải thích lý do** và **nêu cách kiểm tra**, rồi tự kiểm tra trước khi dùng. Mỗi prompt làm thay đổi bài phải ghi vào AI Audit Log bằng hàm `audit(...)` ở ô trên. Mẫu prompt theo bước (thay `{TOPIC}`, `{cột}` bằng thông tin thật của nhóm):

| Bước | Mẫu prompt | Cần tự kiểm tra gì |
|---|---|---|
| Thu thập | "Đề tài của tôi là {TOPIC}. Nguồn chính là {PRIMARY_SOURCE}. Hãy gợi ý 3 nguồn phụ công khai (thời tiết, sự kiện, giá, lịch) có thể ghép theo cột {TIME_COL} và {UNIT_COL}, kèm URL tải và giấy phép." | Mở URL, xác nhận có tải được và giấy phép cho phép dùng |
| So sánh nguồn | "Tôi có hai nguồn về cùng biến {TARGET_COL} ở tần suất {FREQ}. Hãy đề xuất 4 chỉ số để so sánh độ phủ, độ trễ cập nhật và độ lệch giữa hai nguồn." | Chạy lại số trên dữ liệu thật, không dùng số AI đưa |
| Làm sạch | "Cột {cột} có {x}% thiếu, thiếu theo cụm dài nhất {n} bước. Nên nội suy hay bỏ? Giải thích rủi ro rò rỉ tương lai." | Kiểm tra nội suy không dùng giá trị tương lai |
| SQL | "Viết truy vấn DuckDB tạo đặc trưng trễ {SEASON} bước và trung bình trượt theo {UNIT_COL}, có WINDOW, không rò rỉ." | Đếm dòng, kiểm tra cột NULL ở đầu chuỗi |
| EDA | "Từ bảng thống kê sau (dán bảng), nêu 3 nhận xét có thể kiểm chứng và 2 điều cần cảnh giác." | Mỗi nhận xét phải chỉ ra được ô số liệu tương ứng |
| Lỗi | "Lỗi: {dán nguyên văn}. Ngữ cảnh: {ô code}. Nguyên nhân và cách sửa tối thiểu?" | Sửa xong chạy lại ô test |

Ví dụ ghi Audit Log sau khi hỏi: `audit("Thu thập", "Đề tài ... gợi ý nguồn phụ", "Claude", "3 nguồn: NOAA ISD, ...", "mở 3 URL, 1 URL lỗi 404", "dùng NOAA ISD, loại nguồn 404", hallucination=True)`.

## Bước 0: môi trường và cấu trúc thư mục

**Làm gì**: tạo môi trường ảo một lần, cài thư viện, tạo thư mục chuẩn. **Vì sao**: cả hai sinh viên và giảng viên chạy cùng phiên bản thư viện thì kết quả tái tạo được; `requirements.txt` là bằng chứng.

**Cấu trúc**: `data/raw` (file gốc, không sửa), `data/processed` (parquet sạch), `sql/` (truy vấn), `report/` (bảng, hình, nhật ký), `notebooks/`.

In [18]:
# Step 0a: run once in a terminal (not in the notebook)
# python -m venv .venv && source .venv/bin/activate      (Windows: .venv\Scripts\activate)
# pip install pandas numpy duckdb pyarrow scikit-learn lightgbm statsmodels matplotlib seaborn ydata-profiling mapie shap requests
# pip freeze > requirements.txt
import os
for d in ["data/raw", "data/processed", "sql", "report", "notebooks"]: os.makedirs(d, exist_ok=True)
print("folders ready")

folders ready


In [19]:
# Step 0b: imports and plotting style (English labels in figures, no top/right spines, no in-figure titles)
import pandas as pd, numpy as np, duckdb, json, hashlib, warnings, re
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore"); np.random.seed(42)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
TEAL, ACC, GREY = "#1B6B6D", "#F4A261", "#9FBFBF"
def style(ax): ax.spines[["top", "right"]].set_visible(False)
def savefig(fig, name): fig.tight_layout(); fig.savefig(f"report/{name}.png", dpi=300); plt.close(fig); print("saved report/" + name + ".png")
con = duckdb.connect()
print("pandas", pd.__version__, "duckdb", duckdb.__version__)

pandas 2.3.3 duckdb 1.5.5


## Bước 1: bài toán, nguồn dữ liệu và kế hoạch thu thập

**Làm gì**: viết vấn đề bằng 5 câu, đăng ký hai nguồn (chính và phụ) vào sổ nguồn (`report/data_sources.csv`) với URL, giấy phép, ngày tải, số dòng. **Vì sao**: phần Data của bài báo cần ghi rõ nguồn là công trình nào, năm nào, số dòng, số cột và lý do chọn; sổ nguồn giúp viết phần đó trong hai phút.

**Nguồn phụ nên chọn**: biến có cơ chế ảnh hưởng rõ đến mục tiêu (thời tiết với nhu cầu, giá với sạc, sự kiện với lưu lượng), có cùng hoặc cao hơn tần suất mục tiêu, và có thể ghép theo thời gian và đơn vị.

In [20]:
# Step 1a: source registry (append every file you download; this becomes Table "Data sources" in the paper)
SOURCES_PATH = "report/data_sources.csv"
def register_source(src, rows=None, cols=None, start=None, end=None, note=""):
    """src: dict with name, url, license, path. rows/cols/start/end are filled after loading."""
    row = {**src, "downloaded": pd.Timestamp.today().date().isoformat(), "rows": rows, "cols": cols, "start": start, "end": end, "note": note}
    df = pd.DataFrame([row]); df.to_csv(SOURCES_PATH, mode="a", header=not os.path.exists(SOURCES_PATH), index=False)
    return row
print("registry at", SOURCES_PATH)

registry at report/data_sources.csv


In [21]:
def load_table(path):
    if path.endswith(".parquet"):
        return con.execute(f"SELECT * FROM read_parquet('{path}')").df()
    return con.execute(
        f"SELECT * FROM read_csv_auto('{path}', union_by_name=true, sample_size=-1)"
    ).df()

raw = load_table(PRIMARY_SOURCE["path"])

raw.columns = [c.strip() for c in raw.columns]

raw[TIME_COL] = pd.to_datetime(
    raw[TIME_COL],
    errors="coerce",
    utc=False
)

print(raw.shape)
display(raw.head(3))
display(raw.dtypes.value_counts())

register_source(
    PRIMARY_SOURCE,
    rows=len(raw),
    cols=raw.shape[1],
    start=str(raw[TIME_COL].min()),
    end=str(raw[TIME_COL].max())
)

(36857, 3)


,site,ts,kwh
0,caltech,2018-04-25 11:00:00,2.644000
1,caltech,2018-04-25 12:00:00,2.644000
2,caltech,2018-04-25 13:00:00,3.956326


object            1
datetime64[us]    1
float64           1
Name: count, dtype: int64

{'name': 'ACN-Data EV charging',
 'url': 'https://ev.caltech.edu/dataset',
 'license': 'research/public dataset',
 'path': 'data/raw/acn_hourly.csv',
 'downloaded': '2026-09-19',
 'rows': 36857,
 'cols': 3,
 'start': '2018-04-25 11:00:00',
 'end': '2021-09-14 14:00:00',
 'note': ''}

In [78]:
# Step 1c: load the secondary source (covariates); it must share TIME_COL (and UNIT_COL if it is unit-specific)
sec = load_table(SECOND_SOURCE["path"]); sec.columns = [c.strip() for c in sec.columns]
sec[TIME_COL] = pd.to_datetime(sec[TIME_COL], errors="coerce")
SEC_HAS_UNIT = UNIT_COL in sec.columns
print(sec.shape, "unit-specific:", SEC_HAS_UNIT); display(sec.head(3))
register_source(SECOND_SOURCE, rows=len(sec), cols=sec.shape[1], start=str(sec[TIME_COL].min()), end=str(sec[TIME_COL].max()))

(35050, 5) unit-specific: False


,ts,temperature,humidity,wind_speed,precipitation
0,2018-01-01 08:00:00,9.4,66.0,2.6,0.0
1,2018-01-01 09:00:00,9.4,69.0,0.0,0.0
2,2018-01-01 10:00:00,8.9,68.0,0.0,0.0


{'name': 'NOAA Pasadena hourly weather',
 'url': 'https://www.ncei.noaa.gov/',
 'license': 'public data',
 'path': 'data/raw/pasadena_hourly.csv',
 'downloaded': '2026-09-19',
 'rows': 35050,
 'cols': 5,
 'start': '2018-01-01 08:00:00',
 'end': '2022-01-01 07:00:00',
 'note': ''}

### Xuất và đọc lại dữ liệu bằng CSV

**Quy ước file**: mọi bước lưu một file CSV riêng để giảng viên và bạn cùng nhóm mở được bằng Excel mà không cần Python: `data/raw/primary_snapshot.csv` (bản gốc đã chuẩn tên cột và kiểu thời gian, không sửa giá trị), `data/processed/clean.csv` (sau làm sạch, một dòng một đơn vị-thời điểm), `data/processed/feat.csv` (bảng đặc trưng bàn giao). Parquet vẫn lưu song song vì nhanh và giữ đúng kiểu dữ liệu; CSV là bản để đọc và nộp. Khi đọc lại CSV, luôn ép kiểu thời gian bằng `parse_dates` và kiểm tra số dòng trùng khớp với manifest.

In [25]:
# Export the standardised raw snapshot to CSV and read it back to confirm the round trip (dtypes and row count)
RAW_CSV = "data/raw/primary_snapshot.csv"
raw.to_csv(RAW_CSV, index=False)
check = pd.read_csv(RAW_CSV, parse_dates=[TIME_COL])
assert len(check) == len(raw), "row count changed in the CSV round trip"
print("wrote", RAW_CSV, "| rows", len(check), "| columns", list(check.columns)[:8])

wrote data/raw/primary_snapshot.csv | rows 36857 | columns ['site', 'ts', 'kwh']


### Bước 1d: thu thập thêm dữ liệu (mở rộng phạm vi thời gian hoặc đơn vị)

**Làm gì**: nếu nguồn cho phép, tải thêm giai đoạn hoặc thêm đơn vị (trạm, vùng) để bảng đủ khoảng 100 nghìn dòng và có đủ ít nhất hai mùa hoặc hai chế độ. Hàm dưới ghép nhiều file tải theo năm hoặc theo đơn vị và loại trùng. **Vì sao**: mô hình toàn cục và kiểm thử đơn vị chưa thấy cần nhiều đơn vị; đổi chế độ chỉ đo được khi có dữ liệu trước và sau.

In [26]:
# Step 1d: combine several downloaded chunks (data/raw/primary_*.csv) into one table, deduplicated on unit and time
import glob
chunks = sorted(glob.glob(PRIMARY_SOURCE["path"].replace(".csv", "_*.csv")))
if chunks:
    extra = pd.concat([load_table(f) for f in chunks]); extra[TIME_COL] = pd.to_datetime(extra[TIME_COL], errors="coerce")
    before = len(raw); raw = pd.concat([raw, extra]).drop_duplicates([UNIT_COL, TIME_COL] if UNIT_COL in raw.columns else [TIME_COL])
    print(f"added {len(raw) - before} rows from {len(chunks)} chunks")
else:
    print("no extra chunks found (pattern primary_*.csv); skip if your source is a single file")
if UNIT_COL not in raw.columns: raw[UNIT_COL] = "all"; print("single-series data: UNIT_COL set to 'all'")

no extra chunks found (pattern primary_*.csv); skip if your source is a single file


## Bước 2: so sánh hai nguồn và làm sạch có nhật ký

**So sánh nguồn (RQ1a)**: trước khi ghép, đo (1) khoảng thời gian chồng nhau, (2) tỷ lệ bước thời gian có ở nguồn này mà không có ở nguồn kia, (3) nếu hai nguồn có cùng một biến thì tương quan và độ lệch trung bình, (4) độ trễ cập nhật (thời điểm cuối của mỗi nguồn). Bảng này vào phần Data của bài.

**Làm sạch có nhật ký**: mỗi thao tác ghi số dòng trước, sau và lý do; bảng nhật ký là bằng chứng để người đọc tin dữ liệu.

In [27]:
# Step 2a: source comparison table
def compare_sources(a, b, time_col, common_var=None):
    ta, tb = a[time_col].dropna(), b[time_col].dropna()
    ov_start, ov_end = max(ta.min(), tb.min()), min(ta.max(), tb.max())
    ga = set(ta.dt.floor(FREQ).unique()); gb = set(tb.dt.floor(FREQ).unique())
    rows = [["overlap start", ov_start], ["overlap end", ov_end], ["steps only in primary", len(ga - gb)], ["steps only in secondary", len(gb - ga)],
            ["latest timestamp primary", ta.max()], ["latest timestamp secondary", tb.max()]]
    if common_var and common_var in a.columns and common_var in b.columns:
        m = a.groupby(time_col)[common_var].mean().to_frame("a").join(b.groupby(time_col)[common_var].mean().to_frame("b"), how="inner")
        rows += [["correlation of common variable", round(m.a.corr(m.b), 3)], ["mean difference (primary minus secondary)", round((m.a - m.b).mean(), 3)]]
    return pd.DataFrame(rows, columns=["metric", "value"])
cmp_tbl = compare_sources(raw, sec, TIME_COL, common_var=None)     # set common_var="temp" if both sources carry the same variable
cmp_tbl.to_csv("report/table_source_comparison.csv", index=False); display(cmp_tbl)

,metric,value
0,overlap start,2018-04-25 11:00:00
1,overlap end,2021-09-14 14:00:00
2,steps only in primary,7
3,steps only in secondary,11240
4,latest timestamp primary,2021-09-14 14:00:00
5,latest timestamp secondary,2021-12-31 23:00:00


In [79]:
# Step 2b: cleaning with a log; each step records rows before and after and the reason
clean_log = []
def step(df, name, fn, reason):
    n0 = len(df); out = fn(df); clean_log.append([name, n0, len(out), n0 - len(out), reason]); return out
df = raw.copy()
df = step(df, "parse time", lambda x: x.dropna(subset=[TIME_COL]), "rows with unparseable timestamps")
df = step(df, "drop duplicates", lambda x: x.drop_duplicates([UNIT_COL, TIME_COL]), "same unit and timestamp")
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = step(df, "drop missing target", lambda x: x.dropna(subset=[TARGET_COL]), "target missing")
lo, hi = df[TARGET_COL].quantile([0.001, 0.999])
df = step(df, "physical range", lambda x: x[(x[TARGET_COL] >= lo) & (x[TARGET_COL] <= hi)], f"outside [{lo:.3g}, {hi:.3g}] (0.1% tails; replace with a physical range if you know it)")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
display(pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]))

,step,rows_before,rows_after,rows_removed,reason
0,parse time,36857,36857,0,rows with unparseable timestamps
1,drop duplicates,36857,36857,0,same unit and timestamp
2,drop missing target,36857,36857,0,target missing
3,physical range,36857,36786,71,"outside [0.0281, 120] (0.1% tails; replace wit..."


In [80]:
# Step 2c adapted for EV charging load:
# missing hours inside each site's observed period mean no charging -> 0 kWh

parts = []

for site, g in df.groupby(UNIT_COL):
    g = g.sort_values(TIME_COL).set_index(TIME_COL)

    full_idx = pd.date_range(
        g.index.min(),
        g.index.max(),
        freq=FREQ
    )

    g = g.reindex(full_idx)
    g.index.name = TIME_COL
    g[UNIT_COL] = site

    # No recorded charging session in that hour = zero site charging load
    g[TARGET_COL] = g[TARGET_COL].fillna(0)

    parts.append(g.reset_index())

df = pd.concat(parts, ignore_index=True)

clean_log.append([
    "regular hourly grid + zero-fill",
    36786,
    len(df),
    0,
    "hours without charging sessions represent zero EV charging load"
])

print("Rows after regular hourly grid:", len(df))
print(df.groupby(UNIT_COL).size())
print("Missing target:", df[TARGET_COL].isna().sum())

Rows after regular hourly grid: 56229
site
caltech    29705
jpl        26524
dtype: int64
Missing target: 0


In [81]:
# Step 2d adapted for this dataset:
# keep both Caltech and JPL

cnt = df.groupby(UNIT_COL).size()
keep = cnt.index

n_before = len(df)
df = df[df[UNIT_COL].isin(keep)]

clean_log.append([
    "keep available units",
    n_before,
    len(df),
    n_before - len(df),
    "both Caltech and JPL retained because both have substantial hourly history"
])

print("units kept:", len(keep), "of", len(cnt))
print(cnt)

units kept: 2 of 2
site
caltech    29705
jpl        26524
dtype: int64


In [82]:
# Step 2e: merge weather with matching datetime precision

sec_num = sec[
    [TIME_COL]
    + ([UNIT_COL] if SEC_HAS_UNIT else [])
    + [c for c in EXOG_COLS if c in sec.columns]
].copy()

for c in EXOG_COLS:
    if c in sec_num.columns:
        sec_num[c] = pd.to_numeric(sec_num[c], errors="coerce")

# Make both timestamp columns exactly datetime64[ns]
df[TIME_COL] = pd.to_datetime(df[TIME_COL]).astype("datetime64[ns]")
sec_num[TIME_COL] = pd.to_datetime(sec_num[TIME_COL]).astype("datetime64[ns]")

# Sort before merge_asof
df = df.sort_values(TIME_COL)
sec_num = sec_num.sort_values(TIME_COL)

tol = pd.Timedelta(pd.tseries.frequencies.to_offset(FREQ))

m = pd.merge_asof(
    df,
    sec_num,
    on=TIME_COL,
    by=UNIT_COL if SEC_HAS_UNIT else None,
    direction="backward",
    tolerance=tol
)

share = {
    c: round(m[c].notna().mean(), 3)
    for c in EXOG_COLS
    if c in m.columns
}

print("df time dtype:", df[TIME_COL].dtype)
print("weather time dtype:", sec_num[TIME_COL].dtype)
print("share of rows with each covariate:", share)

df = m.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

df time dtype: datetime64[ns]
weather time dtype: datetime64[ns]
share of rows with each covariate: {'temperature': np.float64(1.0), 'humidity': np.float64(1.0), 'wind_speed': np.float64(0.998), 'precipitation': np.float64(0.993)}


In [84]:
# Step 2f: save the clean table and the cleaning log; print a one-paragraph data statement for the paper
df.to_parquet("data/processed/clean.parquet", index=False)
df.to_csv("data/processed/clean.csv", index=False)                     # separate clean CSV for submission and for opening in Excel
chk = pd.read_csv("data/processed/clean.csv", parse_dates=[TIME_COL]); assert len(chk) == len(df) and chk[TARGET_COL].notna().all(), "clean.csv round trip failed"
print("wrote data/processed/clean.csv with", len(chk), "rows")
pd.DataFrame(clean_log, columns=["step", "rows_before", "rows_after", "rows_removed", "reason"]).to_csv("report/table_cleaning_log.csv", index=False)
print(f"Data statement: {PRIMARY_SOURCE['name']} ({PRIMARY_SOURCE['license']}) merged with {SECOND_SOURCE['name']}; {len(df):,} rows, {df[UNIT_COL].nunique()} units, "
      f"{df[TIME_COL].min().date()} to {df[TIME_COL].max().date()} at frequency {FREQ}; {sum(r[3] for r in clean_log):,} rows removed by cleaning.")

wrote data/processed/clean.csv with 56229 rows
Data statement: ACN-Data EV charging (research/public dataset) merged with NOAA Pasadena hourly weather; 56,229 rows, 2 units, 2018-04-25 to 2021-09-14 at frequency h; 71 rows removed by cleaning.


## EDA (RQ1): mô tả, mùa vụ, đơn vị, nguồn phụ, tự tương quan

**Cách đọc**: mỗi hình trả lời một câu hỏi nhỏ và ghi một nhận xét có thể kiểm chứng trong caption. Không vẽ hình chỉ để đẹp. Thứ tự: (1) tổng quan phân bố, (2) chuỗi theo thời gian của vài đơn vị, (3) mùa vụ theo lịch, (4) khác biệt giữa đơn vị, (5) quan hệ với nguồn phụ và độ trễ, (6) tự tương quan quyết định đặc trưng trễ, (7) thiếu dữ liệu.

In [85]:
# EDA 1: automatic profile and summary statistics
from ydata_profiling import ProfileReport

sample = df.sample(
    min(20000, len(df)),
    random_state=42
)

ProfileReport(
    sample,
    title=f"Profile {GROUP}",
    minimal=True
).to_file("report/profile.html")

desc = df[
    [TARGET_COL] +
    [c for c in EXOG_COLS if c in df.columns]
].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).T.round(3)

desc.to_csv("report/table_describe.csv")

display(desc)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:00<00:00, 317.79it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

,count,mean,std,min,1%,5%,50%,95%,99%,max
kwh,56229.0,13.496,22.888,0.0,0.0,0.0,2.822,73.924,98.571,120.396
temperature,56211.0,18.429,6.606,1.7,6.1,8.3,17.800,30.600,34.400,45.600
humidity,56209.0,55.935,23.059,2.0,7.0,15.0,58.000,90.000,96.000,100.000
wind_speed,56135.0,2.394,1.862,0.0,0.0,0.0,2.100,5.700,7.700,16.000
precipitation,55825.0,0.028,0.307,0.0,0.0,0.0,0.000,0.000,0.800,18.300


In [35]:
# EDA 2: distribution of the target (histogram with many bins) and log-scale check for skewed data
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(df[TARGET_COL], bins=60, color=TEAL); axes[0].set_xlabel(TARGET_COL); axes[0].set_ylabel("Count"); style(axes[0])
pos = df[TARGET_COL][df[TARGET_COL] > 0]
axes[1].hist(np.log1p(pos), bins=60, color=ACC); axes[1].set_xlabel("log1p(" + TARGET_COL + ")"); style(axes[1])
savefig(fig, "fig_eda_distribution"); print("skewness:", round(df[TARGET_COL].skew(), 2), "-> consider modelling log1p if skewness > 2")

saved report/fig_eda_distribution.png
skewness: 2.28 -> consider modelling log1p if skewness > 2


In [36]:
# EDA 3: time series of the first four units over the whole period (look for trends, regime changes, gaps)
units = df[UNIT_COL].unique()[:4]
fig, ax = plt.subplots(figsize=(11, 3.4))
for u, col in zip(units, [TEAL, ACC, GREY, "#264653"]):
    g = df[df[UNIT_COL] == u]; ax.plot(g[TIME_COL], g[TARGET_COL], lw=0.7, color=col, label=str(u))
ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--", lw=1); ax.set_ylabel(TARGET_COL); ax.legend(frameon=False, ncol=4); style(ax)
savefig(fig, "fig_eda_timeseries")

saved report/fig_eda_timeseries.png


In [39]:
# EDA 4: calendar seasonality: mean target by hour of day, day of week and month (only the ones that make sense at your FREQ)
t = df[TIME_COL]; cal = pd.DataFrame({"hour": t.dt.hour, "dow": t.dt.dayofweek, "month": t.dt.month, TARGET_COL: df[TARGET_COL]})
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, k in zip(axes, ["hour", "dow", "month"]):
    s = cal.groupby(k)[TARGET_COL].mean(); ax.plot(s.index, s.values, marker="o", color=TEAL); ax.set_xlabel(k); style(ax)
axes[0].set_ylabel("mean " + TARGET_COL); savefig(fig, "fig_eda_seasonality")
season_strength = {k: round(cal.groupby(k)[TARGET_COL].mean().std() / cal[TARGET_COL].std(), 3) for k in ["hour", "dow", "month"]}
print("seasonal strength (std of group means / total std):", season_strength)

saved report/fig_eda_seasonality.png
seasonal strength (std of group means / total std): {'hour': np.float64(0.477), 'dow': np.float64(0.27), 'month': np.float64(0.067)}


In [40]:
# EDA 5: heterogeneity across units: mean and variability per unit, sorted (global models must handle this spread)
per_unit = df.groupby(UNIT_COL)[TARGET_COL].agg(["mean", "std", "min", "max", "count"]).sort_values("mean")
per_unit.to_csv("report/table_per_unit.csv")
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.bar(range(len(per_unit)), per_unit["mean"], color=TEAL, yerr=None)
ax.set_xlabel("units sorted by mean"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_units")
print("ratio of largest to smallest unit mean:", round(per_unit["mean"].max() / max(per_unit["mean"].min(), 1e-9), 2))

saved report/fig_eda_units.png
ratio of largest to smallest unit mean: 1.8


In [86]:
# EDA 6: relation with covariates: binned means and Spearman correlation (robust to outliers)
from scipy.stats import spearmanr
rows = []
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, p = spearmanr(ok[c], ok[TARGET_COL]); rows.append([c, round(rho, 3), p, len(ok)])
    fig, ax = plt.subplots(figsize=(5.5, 3.2)); b = ok.groupby(pd.qcut(ok[c], 10, duplicates="drop"))[TARGET_COL].mean()
    ax.plot(range(len(b)), b.values, marker="o", color=TEAL); ax.set_xlabel(c + " (deciles)"); ax.set_ylabel("mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_covariate_" + c)
corr_tbl = pd.DataFrame(rows, columns=["covariate", "spearman_rho", "p_value", "n"]); corr_tbl.to_csv("report/table_covariate_corr.csv", index=False); display(corr_tbl)

saved report/fig_eda_covariate_temperature.png
saved report/fig_eda_covariate_humidity.png
saved report/fig_eda_covariate_wind_speed.png
saved report/fig_eda_covariate_precipitation.png


,covariate,spearman_rho,p_value,n
0,temperature,0.204,0.000000,56211
1,humidity,-0.160,0.000000,56209
2,wind_speed,0.218,0.000000,56135
3,precipitation,-0.004,0.383755,55825


In [87]:
# EDA 7: cross-correlation with lags: does the covariate lead the target? (positive lag = covariate earlier)
def cross_corr(g, x, y, lags):
    return [g[y].corr(g[x].shift(k)) for k in lags]
lags = list(range(0, 4 * SEASON + 1, max(1, SEASON // 4)))
fig, ax = plt.subplots(figsize=(7, 3.2))
for c in [c for c in EXOG_COLS if c in df.columns][:3]:
    cc = np.nanmean([cross_corr(g, c, TARGET_COL, lags) for _, g in df.groupby(UNIT_COL) if len(g) > 5 * SEASON], axis=0)
    ax.plot(lags, cc, marker="o", label=c); print(c, "best lag:", lags[int(np.nanargmax(np.abs(cc)))], "corr:", round(float(np.nanmax(np.abs(cc))), 3))
ax.axhline(0, color=GREY); ax.set_xlabel("lag (steps, covariate leads target)"); ax.set_ylabel("correlation"); ax.legend(frameon=False); style(ax); savefig(fig, "fig_eda_crosscorr")

temperature best lag: 72 corr: 0.25
humidity best lag: 36 corr: 0.297
wind_speed best lag: 24 corr: 0.209
saved report/fig_eda_crosscorr.png


In [83]:
print("Secondary time range:")
print(sec[TIME_COL].min(), "->", sec[TIME_COL].max())

print("\nFirst 3 weather rows:")
display(sec[[TIME_COL] + EXOG_COLS].head(3))

print("\nFirst merged EV rows:")
display(
    df[[UNIT_COL, TIME_COL, TARGET_COL] + EXOG_COLS].head(5)
)

Secondary time range:
2018-01-01 08:00:00 -> 2022-01-01 07:00:00

First 3 weather rows:


,ts,temperature,humidity,wind_speed,precipitation
0,2018-01-01 08:00:00,9.4,66.0,2.6,0.0
1,2018-01-01 09:00:00,9.4,69.0,0.0,0.0
2,2018-01-01 10:00:00,8.9,68.0,0.0,0.0



First merged EV rows:


,site,ts,kwh,temperature,humidity,wind_speed,precipitation
0,caltech,2018-04-25 11:00:00,2.644000,12.10,98.5,1.50,0.0
1,caltech,2018-04-25 12:00:00,2.644000,12.20,97.0,2.35,0.0
2,caltech,2018-04-25 13:00:00,3.956326,11.95,96.5,2.35,0.0
3,caltech,2018-04-25 14:00:00,7.718726,12.10,93.5,0.75,0.0
4,caltech,2018-04-25 15:00:00,16.894355,13.60,87.0,1.50,0.0


In [43]:
# EDA 8: autocorrelation of the target (ACF) averaged over units: which lags to use as features and whether SEASON is right
from statsmodels.tsa.stattools import acf
nl = 2 * SEASON + 1
acfs = [acf(g[TARGET_COL].values, nlags=nl, fft=True) for _, g in df.groupby(UNIT_COL) if len(g) > 4 * SEASON]
acf_mean = np.mean(acfs, axis=0)
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.stem(range(nl + 1), acf_mean, basefmt=" "); ax.set_xlabel("lag (steps)"); ax.set_ylabel("ACF"); style(ax); savefig(fig, "fig_eda_acf")
print("ACF at lag 1:", round(acf_mean[1], 3), "| at SEASON:", round(acf_mean[SEASON], 3), "-> a seasonal naive baseline is strong if ACF at SEASON > 0.7")

saved report/fig_eda_acf.png
ACF at lag 1: 0.932 | at SEASON: 0.728 -> a seasonal naive baseline is strong if ACF at SEASON > 0.7


In [44]:
# EDA 9: missingness pattern over time per unit (heatmap of monthly missing share) and regime check (rolling mean over time)
miss = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("MS").apply(lambda s: s.isna().mean()).unstack(0)
fig, ax = plt.subplots(figsize=(10, 3.4)); sns.heatmap(miss.T, cmap="Blues", cbar_kws={"label": "missing share"}, ax=ax); ax.set_xlabel("month"); ax.set_ylabel(UNIT_COL); savefig(fig, "fig_eda_missing")
roll = df.set_index(TIME_COL)[TARGET_COL].resample("MS").mean()
fig, ax = plt.subplots(figsize=(10, 3.0)); ax.plot(roll.index, roll.values, color=TEAL, marker="o", ms=3); ax.axvline(pd.Timestamp(TEST_START), color="k", ls="--"); ax.set_ylabel("monthly mean " + TARGET_COL); style(ax); savefig(fig, "fig_eda_regime")
print("largest month-to-month change (share of mean):", round((roll.diff().abs().max() / roll.mean()), 3), "-> above 0.5 suggests a regime shift worth reporting in RQ3")

saved report/fig_eda_missing.png
saved report/fig_eda_regime.png
largest month-to-month change (share of mean): 0.656 -> above 0.5 suggests a regime shift worth reporting in RQ3


### Ghi nhận xét EDA (bắt buộc, dùng cho phần Results RQ1)

Điền 5 nhận xét, mỗi nhận xét chỉ đúng một con số ở bảng hoặc hình phía trên, ví dụ: "Sức mạnh mùa theo giờ là 0,42 trong khi theo tháng chỉ 0,08, nên đặc trưng giờ quan trọng hơn tháng (Hình EDA 4)". Không viết nhận xét không có số.

In [88]:
# EDA 10: updated evidence-backed observations

eda_notes = [
    "1. Phân phối tải sạc EV lệch phải khá mạnh với skewness = 2.28, vì vậy có thể cân nhắc biến đổi log1p(kwh) khi xây dựng mô hình.",

    "2. Mùa vụ theo giờ trong ngày mạnh nhất với seasonal strength = 0.477, cao hơn theo thứ trong tuần (0.270) và theo tháng (0.067).",

    "3. Mức tải sạc trung bình giữa hai site có khác biệt: tỷ lệ giữa site có mean lớn nhất và nhỏ nhất là 1.8.",

    "4. Sau khi chuẩn hóa thời gian NOAA về UTC, wind_speed có tương quan Spearman lớn nhất với tải sạc (rho = 0.218), trong khi precipitation gần như không có tương quan (rho = -0.004, p = 0.384).",

    "5. Tải sạc có tính phụ thuộc thời gian mạnh với ACF tại lag 1 = 0.932 và tại chu kỳ 24 giờ = 0.728; thay đổi lớn nhất giữa hai tháng liên tiếp là 0.656."
]

with open("report/eda_notes.md", "w", encoding="utf-8") as f:
    f.write("\n".join(eda_notes))

print("updated report/eda_notes.md")

updated report/eda_notes.md


## Bước 3: SQL với DuckDB, trả lời RQ1 và tạo bảng đặc trưng

**Vì sao SQL**: người đọc bài kiểm tra được từng con số bằng một truy vấn; đặc trưng trễ và cửa sổ trượt bằng `WINDOW` không rò rỉ tương lai nếu chỉ dùng `LAG` và `PRECEDING`.

**Quy tắc chống rò rỉ**: đặc trưng chỉ dùng thông tin **tại hoặc trước** thời điểm t; mục tiêu là `LEAD(target, h)`; không dùng thống kê tính trên toàn chuỗi (kể cả trung bình đơn vị) trừ khi tính chỉ trên phần huấn luyện.

In [98]:
# Step 3a: register the clean table and write the RQ1 queries to sql/queries.sql (edit the four queries to your topic; keep them minimal and readable)
con.register("clean", df)
EXOG_SQL = ", ".join([c for c in EXOG_COLS if c in df.columns]) or "NULL AS no_exog"
SQL_RQ1 = f"""
-- Q1: mean and spread of the target per unit and per calendar period
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS mean_target, STDDEV({TARGET_COL}) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the {EVENT_QUANTILE:.0%} quantile) per unit
WITH thr AS (SELECT quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) AS q FROM clean)
SELECT {UNIT_COL}, AVG(({TARGET_COL} > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG({TARGET_COL}) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY {EXOG_COLS[0] if EXOG_COLS else TARGET_COL}) AS decile, {TARGET_COL} FROM clean WHERE {EXOG_COLS[0] if EXOG_COLS else TARGET_COL} IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON steps earlier
SELECT corr({TARGET_COL}, lagged) AS r_season FROM (SELECT {TARGET_COL}, LAG({TARGET_COL}, {SEASON}) OVER (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL}) AS lagged FROM clean);
"""
open("sql/queries.sql", "w").write(SQL_RQ1); print(SQL_RQ1)


-- Q1: mean and spread of the target per unit and per calendar period
SELECT site, month(ts) AS mon, AVG(kwh) AS mean_target, STDDEV(kwh) AS sd_target, COUNT(*) AS n FROM clean GROUP BY 1, 2 ORDER BY 1, 2;
-- Q2: share of event steps (target above the 90% quantile) per unit
WITH thr AS (SELECT quantile_cont(kwh, 0.9) AS q FROM clean)
SELECT site, AVG((kwh > q)::INT) AS event_share FROM clean, thr GROUP BY 1 ORDER BY 2 DESC;
-- Q3: target by covariate decile (relationship with the secondary source)
SELECT decile, AVG(kwh) AS mean_target, COUNT(*) AS n FROM (SELECT NTILE(10) OVER (ORDER BY temperature) AS decile, kwh FROM clean WHERE temperature IS NOT NULL) GROUP BY 1 ORDER BY 1;
-- Q4: seasonal-naive skill check: correlation between the target and its value SEASON steps earlier
SELECT corr(kwh, lagged) AS r_season FROM (SELECT kwh, LAG(kwh, 24) OVER (PARTITION BY site ORDER BY ts) AS lagged FROM clean);



In [99]:
# Step 3b: run every statement, save each result as report/table_q{k}.csv (comment lines removed before splitting on ';')
def run_sql_file(path):
    txt = open(path).read(); body = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("--"))
    k = 0
    for stmt in body.split(";"):
        stmt = stmt.strip()
        if not stmt: continue
        k += 1; out = con.execute(stmt).df(); out.to_csv(f"report/table_q{k}.csv", index=False); print(f"Q{k}: {out.shape}"); display(out.head(8))
run_sql_file("sql/queries.sql")

Q1: (24, 5)


,site,mon,mean_target,sd_target,n
0,caltech,1,8.552579,11.677641,2232
1,caltech,2,8.654178,11.355807,2040
2,caltech,3,6.577759,9.232017,2232
3,caltech,4,6.661519,9.871272,2293
4,caltech,5,9.768643,12.773791,2976
5,caltech,6,10.550793,13.361589,2880
6,caltech,7,10.334764,13.922228,2976
7,caltech,8,11.992122,15.407652,2976


Q2: (2, 2)


,site,event_share
0,jpl,0.164568
1,caltech,0.042350


Q3: (10, 3)


,decile,mean_target,n
0,1,5.624433,5622
1,2,8.291479,5621
2,3,10.564235,5621
3,4,12.078732,5621
4,5,12.696815,5621
5,6,12.785518,5621
6,7,14.477177,5621
7,8,17.644250,5621


Q4: (1, 1)


,r_season
0,0.711167


In [100]:
# Step 3c: statistical tests behind RQ1 (report p-values, not only means): Kruskal-Wallis across units, Spearman with covariates
from scipy.stats import kruskal
groups = [g[TARGET_COL].values for _, g in df.groupby(UNIT_COL) if len(g) > 30]
H, p = kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
tests = [["Kruskal-Wallis target across units", round(H, 2) if H == H else "n/a", p]]
for c in [c for c in EXOG_COLS if c in df.columns]:
    ok = df[[c, TARGET_COL]].dropna(); rho, pv = spearmanr(ok[c], ok[TARGET_COL]); tests.append([f"Spearman target vs {c}", round(rho, 3), pv])
tests = pd.DataFrame(tests, columns=["test", "statistic", "p_value"]); tests.to_csv("report/table_rq1_tests.csv", index=False); display(tests)

,test,statistic,p_value
0,Kruskal-Wallis target across units,8.280,0.004007
1,Spearman target vs temperature,0.204,0.000000
2,Spearman target vs humidity,-0.160,0.000000
3,Spearman target vs wind_speed,0.218,0.000000
4,Spearman target vs precipitation,-0.004,0.383755


In [101]:
# Step 3d: build the feature table with SQL (lags, rolling means, calendar, covariates, targets for every horizon); no future information
lag_list = sorted(set([1, 2, 3, SEASON, 2 * SEASON, 7 * SEASON if FREQ in ("h", "H") else SEASON * 4]))
lag_sql = ", ".join([f"LAG({TARGET_COL}, {k}) OVER w AS y_lag{k}" for k in lag_list])
roll_sql = f"AVG({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_ma_season, STDDEV({TARGET_COL}) OVER (w ROWS BETWEEN {SEASON - 1} PRECEDING AND CURRENT ROW) AS y_sd_season, " \
           f"{TARGET_COL} - LAG({TARGET_COL}, 1) OVER w AS y_diff1"
exog_lag = ", ".join([f"LAG({c}, {SEASON}) OVER w AS {c}_lag_season" for c in EXOG_COLS if c in df.columns])
targets = ", ".join([f"LEAD({TARGET_COL}, {h}) OVER w AS y_h{h}" for h in HORIZONS])
cal_sql = f"hour({TIME_COL}) AS hr, dayofweek({TIME_COL}) AS dow, month({TIME_COL}) AS mon, dayofyear({TIME_COL}) AS doy"
FEAT_SQL = f"""
CREATE OR REPLACE TABLE feat AS
SELECT {UNIT_COL}, {TIME_COL}, {TARGET_COL}, {EXOG_SQL}, {cal_sql}, {lag_sql}, {roll_sql}{', ' + exog_lag if exog_lag else ''}, {targets}
FROM clean WINDOW w AS (PARTITION BY {UNIT_COL} ORDER BY {TIME_COL});
"""
con.execute(FEAT_SQL); open("sql/features.sql", "w").write(FEAT_SQL)
feat = con.execute("SELECT * FROM feat").df(); print(feat.shape); display(feat.head(3))

(56229, 26)


,site,ts,kwh,temperature,humidity,wind_speed,precipitation,hr,dow,mon,doy,y_lag1,y_lag2,y_lag3,y_lag24,y_lag48,y_lag168,y_ma_season,y_sd_season,y_diff1,temperature_lag_season,humidity_lag_season,wind_speed_lag_season,precipitation_lag_season,y_h1,y_h24
0,caltech,2019-06-26 03:00:00,4.014417,17.5,79.5,3.6,0.0,3,3,6,177,13.572948,17.108039,19.995712,4.559000,2.71875,6.042516,10.560535,9.592118,-9.558532,17.2,84.0,3.10,0.0,3.171333,4.611796
1,caltech,2019-06-26 04:00:00,3.171333,17.2,81.0,2.1,0.0,4,3,6,177,4.014417,13.572948,17.108039,2.989095,2.71875,2.466071,10.568128,9.585934,-0.843083,16.7,86.0,3.10,0.0,2.718833,3.693425
2,caltech,2019-06-26 05:00:00,2.718833,17.2,81.0,3.1,0.0,5,3,6,177,3.171333,4.014417,13.572948,0.591095,0.00000,0.634571,10.656784,9.499095,-0.452500,16.4,87.0,2.85,0.0,0.000000,1.202425


In [68]:
# Final fix: NOAA uses Pacific Local Standard Time (UTC-8), no DST adjustment

import pandas as pd

weather = pd.read_csv(
    "data/raw/pasadena_hourly.csv",
    parse_dates=["ts"]
)

# NOAA LCD timestamps are Local Standard Time.
# Pacific Standard Time = UTC-8, so convert to UTC by adding 8 hours.
weather["ts"] = (
    pd.to_datetime(weather["ts"])
    + pd.Timedelta(hours=8)
)

weather.to_csv(
    "data/raw/pasadena_hourly.csv",
    index=False
)

print("UTC time range:")
print(weather["ts"].min(), "->", weather["ts"].max())
display(weather.head())

UTC time range:
2018-01-01 08:00:00 -> 2022-01-01 07:00:00


,ts,temperature,humidity,wind_speed,precipitation
0,2018-01-01 08:00:00,9.4,66.0,2.6,0.0
1,2018-01-01 09:00:00,9.4,69.0,0.0,0.0
2,2018-01-01 10:00:00,8.9,68.0,0.0,0.0
3,2018-01-01 11:00:00,8.9,71.0,0.0,0.0
4,2018-01-01 12:00:00,7.8,73.0,0.0,0.0


In [102]:
# Step 3e: leakage check: for every row, the largest lag feature must come from a timestamp strictly before the target timestamp (by construction) and
# the target of horizon h must equal the raw target h steps later within the same unit
g0 = feat[feat[UNIT_COL] == feat[UNIT_COL].iloc[0]].sort_values(TIME_COL).reset_index(drop=True)
h = HORIZONS[0]; ok = np.allclose(g0[f"y_h{h}"].iloc[:-h].values, g0[TARGET_COL].iloc[h:].values, equal_nan=True)
print("horizon target aligned with raw target shifted by h:", ok)
assert ok, "target alignment broken: check FREQ and the resample grid"

horizon target aligned with raw target shifted by h: True


In [103]:
# Step 3f: RQ1 figures 2 and 3 for the paper: event share per unit and target by covariate decile (from the SQL outputs)
q2 = pd.read_csv("report/table_q2.csv"); q3 = pd.read_csv("report/table_q3.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
b = axes[0].bar(range(len(q2)), q2.event_share, color=TEAL); axes[0].set_xlabel("units"); axes[0].set_ylabel("event share"); style(axes[0])
axes[1].plot(q3.decile, q3.mean_target, marker="o", color=ACC); axes[1].set_xlabel("covariate decile"); axes[1].set_ylabel("mean " + TARGET_COL); style(axes[1])
savefig(fig, "fig_rq1_sql")

saved report/fig_rq1_sql.png


## Kiểm tra, bàn giao và ghi Audit Log

**Test case** phải chạy qua trước khi bàn giao. **Manifest** ghi số dòng, cột, khoảng thời gian và mã MD5 của parquet để Sinh viên B xác nhận nhận đúng file. Sau đó điền **AI Audit Log** cho các prompt đã dùng ở tuần 1 và 2 (ít nhất 3 mục mỗi tuần, có ít nhất một mục phát hiện AI sai).

In [105]:
# Sort feature table before final validation
feat = (
    feat.sort_values([UNIT_COL, TIME_COL])
        .reset_index(drop=True)
)

print(
    feat.groupby(UNIT_COL)[TIME_COL]
        .apply(lambda s: s.is_monotonic_increasing)
)

site
caltech    True
jpl        True
Name: ts, dtype: bool


In [106]:
# Test cases for Notebook_C (all asserts must pass)
assert feat[[UNIT_COL, TIME_COL]].duplicated().sum() == 0, "duplicate unit-time rows"
assert feat[TARGET_COL].notna().all(), "target missing after cleaning"
assert set(HORIZONS) <= set(int(c[3:]) for c in feat.columns if c.startswith("y_h")), "horizon targets missing"
assert (feat.groupby(UNIT_COL)[TIME_COL].apply(lambda s: s.is_monotonic_increasing)).all(), "time not sorted within unit"
assert len(feat) >= 30000, f"only {len(feat)} rows: collect more data (Step 1d) or lower FREQ"
print("Tests C: OK")

Tests C: OK


In [107]:
# Handover: parquet + manifest with md5
feat.to_parquet("data/processed/feat.parquet", index=False)
feat.to_csv("data/processed/feat.csv", index=False)                    # CSV copy of the handover table (parquet is the file of record)
md5 = hashlib.md5(open("data/processed/feat.parquet", "rb").read()).hexdigest()
manifest = {"group": GROUP, "topic": TOPIC, "rows": len(feat), "cols": list(feat.columns), "units": int(feat[UNIT_COL].nunique()), "start": str(feat[TIME_COL].min()), "end": str(feat[TIME_COL].max()),
            "freq": FREQ, "horizons": HORIZONS, "season": SEASON, "unit_col": UNIT_COL, "time_col": TIME_COL, "target_col": TARGET_COL, "exog_cols": [c for c in EXOG_COLS if c in feat.columns], "test_start": TEST_START, "md5": md5}
json.dump(manifest, open("data/processed/manifest.json", "w"), indent=2); print(json.dumps(manifest, indent=2)[:600])

{
  "group": "Nhom 11",
  "topic": "D\u1ef1 b\u00e1o t\u1ea3i s\u1ea1c EV theo gi\u1edd",
  "rows": 56229,
  "cols": [
    "site",
    "ts",
    "kwh",
    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",
    "hr",
    "dow",
    "mon",
    "doy",
    "y_lag1",
    "y_lag2",
    "y_lag3",
    "y_lag24",
    "y_lag48",
    "y_lag168",
    "y_ma_season",
    "y_sd_season",
    "y_diff1",
    "temperature_lag_season",
    "humidity_lag_season",
    "wind_speed_lag_season",
    "precipitation_lag_season",
    "y_h1",
    "y_h24"
  ],
  "units": 2,
  "start": "2018-04-25 11:00:


In [56]:
# AI Audit Log - real prompts/actions used for Group 11

audit(
    "CONFIG",
    "Điền CONFIG Notebook C cho đề tài dự báo tải sạc EV theo giờ của Nhóm 11.",
    "ChatGPT",
    "Đề xuất dùng ACN-Data làm nguồn chính, NOAA Pasadena làm nguồn phụ; site/ts/kwh là unit/time/target.",
    "Kiểm tra trực tiếp tên file và các cột trong CSV.",
    "Giữ cấu hình sau khi đối chiếu dữ liệu thật."
)

audit(
    "Step 1",
    "Kiểm tra acn_data.csv và acn_sessions_raw.csv để chọn nguồn chính.",
    "ChatGPT",
    "Khuyến nghị dùng acn_sessions_raw.csv rồi chuyển dữ liệu phiên sạc thành tải theo giờ.",
    "In shape, columns và 3 dòng đầu của cả hai file.",
    "Dùng acn_sessions_raw.csv và tạo acn_hourly.csv."
)

audit(
    "Step 2c",
    "Nên xử lý các giờ không có bản ghi sạc EV như thế nào?",
    "ChatGPT",
    "Khuyến nghị tạo lưới giờ đều và điền 0 kWh cho giờ không có phiên sạc thay vì nội suy.",
    "Kiểm tra ý nghĩa dữ liệu: không có phiên sạc trong giờ tương ứng với tải sạc bằng 0.",
    "Dùng zero-fill cho target trên lưới giờ."
)

audit(
    "Step 2d",
    "Step 2d có nên loại site ngắn hơn 90% site dài nhất không?",
    "ChatGPT",
    "Phát hiện JPL có 26524 giờ so với Caltech 29705 giờ nên sẽ bị loại bởi ngưỡng 90%.",
    "Tính tỷ lệ độ dài giữa hai site và kiểm tra số giờ của từng site.",
    "Giữ cả Caltech và JPL vì cả hai đều có lịch sử đáng kể."
)

audit(
    "Step 2e",
    "Sửa lỗi MergeError datetime64[ns] và datetime64[us] khi merge weather.",
    "ChatGPT",
    "Đề xuất ép cả hai cột ts về datetime64[ns] trước merge_asof.",
    "In dtype của hai cột trước khi merge và chạy lại cell.",
    "Áp dụng ép kiểu datetime64[ns]; merge thành công."
)

audit(
    "Test cases",
    "Sửa lỗi AssertionError: time not sorted within unit.",
    "ChatGPT",
    "Đề xuất sort feat theo site và ts trước final validation.",
    "Kiểm tra is_monotonic_increasing cho Caltech và JPL đều True.",
    "Sort feat trước test; test sau đó chạy qua.",
    hallucination=True
)

print(pd.read_csv(AUDIT_PATH).tail(6))

audit entry saved: CONFIG
audit entry saved: Step 1
audit entry saved: Step 2c
audit entry saved: Step 2d
audit entry saved: Step 2e
audit entry saved: Test cases
         date    group        step                                             prompt     tool                                          ai_output  \
0  2026-09-19  Nhom 11      CONFIG  Điền CONFIG Notebook C cho đề tài dự báo tải s...  ChatGPT  Đề xuất dùng ACN-Data làm nguồn chính, NOAA Pa...   
1  2026-09-19  Nhom 11      Step 1  Kiểm tra acn_data.csv và acn_sessions_raw.csv ...  ChatGPT  Khuyến nghị dùng acn_sessions_raw.csv rồi chuy...   
2  2026-09-19  Nhom 11     Step 2c  Nên xử lý các giờ không có bản ghi sạc EV như ...  ChatGPT  Khuyến nghị tạo lưới giờ đều và điền 0 kWh cho...   
3  2026-09-19  Nhom 11     Step 2d  Step 2d có nên loại site ngắn hơn 90% site dài...  ChatGPT  Phát hiện JPL có 26524 giờ so với Caltech 2970...   
4  2026-09-19  Nhom 11     Step 2e  Sửa lỗi MergeError datetime64[ns] và datetime6...  ChatG

## Bài tập mở rộng cho Sinh viên A (làm ít nhất 3 trong 5, ghi kết quả vào report/exercises_C.md)

1. **Nguồn thứ ba**: thêm một nguồn phụ nữa (lịch ngày lễ, giá, sự kiện) bằng cùng hàm `load_table` và `merge_asof`; báo tỷ lệ dòng ghép được và tương quan với mục tiêu.
2. **So sánh tần suất**: gộp mục tiêu về tần suất thô hơn (ví dụ ngày) và so sức mạnh mùa giữa hai tần suất; kết luận tần suất nào phù hợp cho câu hỏi của đề.
3. **Hai truy vấn SQL thêm**: một truy vấn dùng `QUALIFY` để lấy top 5 đơn vị theo mỗi tháng, một truy vấn `WITH` tính tỷ lệ sự kiện theo mùa; giải thích kết quả bằng hai câu.
4. **Phân rã mùa**: dùng `statsmodels.tsa.seasonal.STL` trên một đơn vị; vẽ trend, seasonal, resid; nhận xét phần dư có còn cấu trúc không.
5. **Kiểm tra rò rỉ có chủ ý**: tạo một đặc trưng sai (dùng `LEAD`) rồi cho Sinh viên B chạy thử để thấy MAE giảm bất thường; ghi lại bài học vào Discussion.

In [57]:
import os
import zipfile
import pandas as pd

caiso_dir = "data/raw/caiso"

files = sorted([
    f for f in os.listdir(caiso_dir)
    if f.endswith(".zip")
])

print("Number of CAISO zip files:", len(files))
print("First file:", files[0])

first_zip = os.path.join(caiso_dir, files[0])

with zipfile.ZipFile(first_zip, "r") as z:
    print("Files inside zip:", z.namelist())

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:
        sample_caiso = pd.read_csv(f)

print("Shape:", sample_caiso.shape)
print("Columns:")
print(sample_caiso.columns.tolist())

display(sample_caiso.head(5))

Number of CAISO zip files: 1461
First file: 20180101_20180101_DAM_LMP_GRP_N_N_v12_csv.zip
Files inside zip: ['20180101_20180101_PRC_LMP_DAM_LMP_v12.csv', '20180101_20180101_PRC_LMP_DAM_MCE_v12.csv', '20180101_20180101_PRC_LMP_DAM_MCC_v12.csv', '20180101_20180101_PRC_LMP_DAM_MCL_v12.csv']
Shape: (249624, 16)
Columns:
['INTERVALSTARTTIME_GMT', 'INTERVALENDTIME_GMT', 'OPR_DT', 'OPR_HR', 'OPR_INTERVAL', 'NODE_ID_XML', 'NODE_ID', 'NODE', 'MARKET_RUN_ID', 'LMP_TYPE', 'XML_DATA_ITEM', 'PNODE_RESMRID', 'GRP_TYPE', 'POS', 'MW', 'GROUP']


,INTERVALSTARTTIME_GMT,INTERVALENDTIME_GMT,OPR_DT,OPR_HR,OPR_INTERVAL,NODE_ID_XML,NODE_ID,NODE,MARKET_RUN_ID,LMP_TYPE,XML_DATA_ITEM,PNODE_RESMRID,GRP_TYPE,POS,MW,GROUP
0,2018-01-01T12:00:00-00:00,2018-01-01T13:00:00-00:00,2018-01-01,5,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,32.08613,1
1,2018-01-01T22:00:00-00:00,2018-01-01T23:00:00-00:00,2018-01-01,15,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,26.39939,1
2,2018-01-01T16:00:00-00:00,2018-01-01T17:00:00-00:00,2018-01-01,9,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,30.57537,1
3,2018-01-01T11:00:00-00:00,2018-01-01T12:00:00-00:00,2018-01-01,4,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,32.02941,1
4,2018-01-01T09:00:00-00:00,2018-01-01T10:00:00-00:00,2018-01-01,2,0,0096WD_7_N001,0096WD_7_N001,0096WD_7_N001,DAM,LMP,LMP_PRC,0096WD_7_N001,ALL,1,35.23226,1


In [58]:
# Exercise 1 - inspect regional CAISO pricing nodes

mask = sample_caiso["NODE"].astype(str).str.contains(
    "SCE|SP15|PASADENA",
    case=False,
    na=False
)

candidates = (
    sample_caiso.loc[
        mask,
        ["NODE", "NODE_ID", "LMP_TYPE", "XML_DATA_ITEM"]
    ]
    .drop_duplicates()
)

print("Number of matching rows:", len(candidates))

display(
    candidates
    .sort_values("NODE")
    .head(50)
)

Number of matching rows: 100


,NODE,NODE_ID,LMP_TYPE,XML_DATA_ITEM
55032,DLAP_SCE-APND,DLAP_SCE-APND,LMP,LMP_PRC
148416,OSCEOLA_2_TNODE,OSCEOLA_2_TNODE,LMP,LMP_PRC
194832,SCEC_1_PDRP01-APND,SCEC_1_PDRP01-APND,LMP,LMP_PRC
194856,SCEC_1_PDRP02-APND,SCEC_1_PDRP02-APND,LMP,LMP_PRC
194880,SCEC_1_PDRP03-APND,SCEC_1_PDRP03-APND,LMP,LMP_PRC
194904,SCEC_1_PDRP15-APND,SCEC_1_PDRP15-APND,LMP,LMP_PRC
194928,SCEC_1_PDRP21-APND,SCEC_1_PDRP21-APND,LMP,LMP_PRC
194952,SCEC_1_PDRP22-APND,SCEC_1_PDRP22-APND,LMP,LMP_PRC
194976,SCEC_1_PDRP23-APND,SCEC_1_PDRP23-APND,LMP,LMP_PRC
195000,SCEC_1_PDRP24-APND,SCEC_1_PDRP24-APND,LMP,LMP_PRC


In [59]:
# Exercise 1 - extract hourly CAISO price for DLAP_SCE-APND

import os
import zipfile
import csv
import pandas as pd

caiso_dir = "data/raw/caiso"
target_node = "DLAP_SCE-APND"

all_zips = sorted([
    f for f in os.listdir(caiso_dir)
    if f.endswith(".zip")
])

# Only read dates overlapping the EV dataset
start_day = pd.Timestamp(df[TIME_COL].min()).strftime("%Y%m%d")
end_day = pd.Timestamp(df[TIME_COL].max()).strftime("%Y%m%d")

use_zips = [
    f for f in all_zips
    if start_day <= f[:8] <= end_day
]

print("CAISO files in EV period:", len(use_zips))

rows = []

for i, fname in enumerate(use_zips, 1):
    path = os.path.join(caiso_dir, fname)

    with zipfile.ZipFile(path, "r") as z:
        # Use only the total DAM LMP file
        lmp_files = [
            n for n in z.namelist()
            if "_PRC_LMP_DAM_LMP_" in n
        ]

        if not lmp_files:
            continue

        csv_name = lmp_files[0]

        with z.open(csv_name) as f:
            # Read header
            header_line = f.readline().decode("utf-8-sig").strip()
            header = next(csv.reader([header_line]))

            idx_ts = header.index("INTERVALSTARTTIME_GMT")
            idx_node = header.index("NODE")
            idx_type = header.index("LMP_TYPE")
            idx_item = header.index("XML_DATA_ITEM")
            idx_price = header.index("MW")

            needle = target_node.encode()

            # Scan raw lines first, parse only matching node
            for line in f:
                if needle not in line:
                    continue

                values = next(
                    csv.reader([line.decode("utf-8-sig").strip()])
                )

                if (
                    values[idx_node] == target_node
                    and values[idx_type] == "LMP"
                    and values[idx_item] == "LMP_PRC"
                ):
                    rows.append([
                        values[idx_ts],
                        values[idx_price]
                    ])

    if i % 100 == 0:
        print(f"Processed {i}/{len(use_zips)} files")

caiso_price = pd.DataFrame(
    rows,
    columns=["ts", "price"]
)

# CAISO timestamp is GMT/UTC
caiso_price["ts"] = (
    pd.to_datetime(
        caiso_price["ts"],
        errors="coerce",
        utc=True
    )
    .dt.tz_localize(None)
)

caiso_price["price"] = pd.to_numeric(
    caiso_price["price"],
    errors="coerce"
)

caiso_price = (
    caiso_price
    .dropna(subset=["ts", "price"])
    .groupby("ts", as_index=False)["price"]
    .mean()
    .sort_values("ts")
)

caiso_price.to_csv(
    "data/raw/caiso_price_hourly.csv",
    index=False
)

print("Shape:", caiso_price.shape)
print(
    "Time:",
    caiso_price["ts"].min(),
    "->",
    caiso_price["ts"].max()
)
print(caiso_price["price"].describe())

display(caiso_price.head())

CAISO files in EV period: 1239
Processed 100/1239 files
Processed 200/1239 files
Processed 300/1239 files
Processed 400/1239 files
Processed 500/1239 files
Processed 600/1239 files
Processed 700/1239 files
Processed 800/1239 files
Processed 900/1239 files
Processed 1000/1239 files
Processed 1100/1239 files
Processed 1200/1239 files
Shape: (29736, 2)
Time: 2018-04-25 07:00:00 -> 2021-09-15 06:00:00
count    29736.000000
mean        41.188350
std         48.527980
min        -15.833060
25%         24.893340
50%         34.292850
75%         46.719498
max       1557.426000
Name: price, dtype: float64


,ts,price
0,2018-04-25 07:00:00,24.22942
1,2018-04-25 08:00:00,22.69083
2,2018-04-25 09:00:00,21.06896
3,2018-04-25 10:00:00,21.25395
4,2018-04-25 11:00:00,23.05009


In [60]:
# Exercise 1: merge CAISO electricity price with EV charging load

from scipy.stats import spearmanr

price = pd.read_csv(
    "data/raw/caiso_price_hourly.csv",
    parse_dates=["ts"]
)

# Make datetime precision identical
price["ts"] = pd.to_datetime(price["ts"]).astype("datetime64[ns]")

df_ex1 = df.copy()
df_ex1["ts"] = pd.to_datetime(df_ex1["ts"]).astype("datetime64[ns]")

# Exact hourly merge
df_ex1 = df_ex1.merge(
    price,
    on="ts",
    how="left"
)

merge_share = df_ex1["price"].notna().mean()

ok = df_ex1[["price", TARGET_COL]].dropna()

rho, p = spearmanr(
    ok["price"],
    ok[TARGET_COL]
)

print("Rows:", len(df_ex1))
print("Price merge share:", round(merge_share, 3))
print("Spearman rho:", round(rho, 3))
print("p-value:", p)

display(
    df_ex1[
        [UNIT_COL, TIME_COL, TARGET_COL, "price"]
    ].head()
)

Rows: 56229
Price merge share: 1.0
Spearman rho: 0.033
p-value: 2.302640928388072e-15


,site,ts,kwh,price
0,caltech,2018-04-25 11:00:00,2.644000,23.05009
1,caltech,2018-04-25 12:00:00,2.644000,43.02700
2,caltech,2018-04-25 13:00:00,3.956326,48.06491
3,caltech,2018-04-25 14:00:00,7.718726,32.04049
4,caltech,2018-04-25 15:00:00,16.894355,24.79371


In [61]:
# Save Exercise 1 result

exercise1_text = """
## Exercise 1 - Third data source: CAISO electricity price

- Third source: CAISO Day-Ahead LMP, node DLAP_SCE-APND.
- Number of EV rows: 56,229.
- Price merge share: 1.000 (100%).
- Spearman correlation between electricity price and EV charging load: rho = 0.033.
- p-value = 2.30e-15.

Conclusion: CAISO electricity price could be matched to all EV observations.
The association with EV charging load is statistically significant but very weak,
so electricity price alone explains little of the variation in hourly charging load.
"""

with open("report/exercises_C.md", "w", encoding="utf-8") as f:
    f.write(exercise1_text)

print("saved Exercise 1 to report/exercises_C.md")

saved Exercise 1 to report/exercises_C.md


In [ ]:
# Exercise 4 starter: STL decomposition for one unit (period = SEASON)
from statsmodels.tsa.seasonal import STL
u0 = df[UNIT_COL].unique()[0]; s = df[df[UNIT_COL] == u0].set_index(TIME_COL)[TARGET_COL].asfreq(FREQ).interpolate(limit=3).dropna()
if len(s) > 3 * SEASON:
    stl = STL(s, period=SEASON, robust=True).fit()
    fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for ax, comp, lab in zip(axes, [stl.trend, stl.seasonal, stl.resid], ["trend", "seasonal", "residual"]): ax.plot(comp, color=TEAL, lw=0.8); ax.set_ylabel(lab); style(ax)
    savefig(fig, "fig_exercise_stl"); print("residual share of variance:", round(float(stl.resid.var() / s.var()), 3))

In [62]:
# Exercise 2 starter: seasonal strength at a coarser frequency (daily) versus the working frequency
daily = df.set_index(TIME_COL).groupby(UNIT_COL)[TARGET_COL].resample("D").mean().reset_index()
strength = lambda frame, key: round(frame.groupby(key)[TARGET_COL].mean().std() / frame[TARGET_COL].std(), 3)
print("weekday strength: working freq", strength(df.assign(k=df[TIME_COL].dt.dayofweek), "k"), "| daily", strength(daily.assign(k=daily[TIME_COL].dt.dayofweek), "k"))

weekday strength: working freq 0.27 | daily 0.446


In [63]:
# Append Exercise 2 result

exercise2_text = """

## Exercise 2 - Compare hourly and daily frequency

- Weekday seasonal strength at hourly frequency: 0.270.
- Weekday seasonal strength at daily frequency: 0.446.

Conclusion: Aggregating the EV charging load to daily frequency makes the
weekday pattern more pronounced. However, hourly frequency is retained for
the main forecasting task because the project aims to forecast hourly EV
charging load and the hour-of-day seasonal effect is also strong.
"""

with open("report/exercises_C.md", "a", encoding="utf-8") as f:
    f.write(exercise2_text)

print("saved Exercise 2 to report/exercises_C.md")

saved Exercise 2 to report/exercises_C.md


In [64]:
# Exercise 3 starter: QUALIFY and CTE examples (edit and add your own two queries to sql/queries.sql)
ex_sql = f"""
SELECT {UNIT_COL}, month({TIME_COL}) AS mon, AVG({TARGET_COL}) AS m, RANK() OVER (PARTITION BY month({TIME_COL}) ORDER BY AVG({TARGET_COL}) DESC) AS rk FROM clean GROUP BY 1, 2 QUALIFY rk <= 5 ORDER BY mon, rk;
"""
display(con.execute(ex_sql).df().head(10))

,site,mon,m,rk
0,jpl,1,21.854358,1
1,caltech,1,8.552579,2
2,jpl,2,22.951457,1
3,caltech,2,8.654178,2
4,jpl,3,18.829079,1
5,caltech,3,6.577759,2
6,jpl,4,15.647635,1
7,caltech,4,6.661519,2
8,jpl,5,16.134628,1
9,caltech,5,9.768643,2


In [65]:
# Exercise 3 - Query 2: CTE for event share by season

sql_ex3_q2 = f"""
WITH base AS (
    SELECT
        *,
        quantile_cont({TARGET_COL}, {EVENT_QUANTILE}) OVER () AS threshold
    FROM clean
),
seasonal AS (
    SELECT
        CASE
            WHEN month({TIME_COL}) IN (12, 1, 2) THEN 'Winter'
            WHEN month({TIME_COL}) IN (3, 4, 5) THEN 'Spring'
            WHEN month({TIME_COL}) IN (6, 7, 8) THEN 'Summer'
            ELSE 'Fall'
        END AS season,
        ({TARGET_COL} > threshold)::INT AS is_event
    FROM base
)
SELECT
    season,
    AVG(is_event) AS event_share,
    COUNT(*) AS n
FROM seasonal
GROUP BY season
ORDER BY season;
"""

ex3_q2 = con.execute(sql_ex3_q2).df()

print("Exercise 3 - Query 2")
display(ex3_q2)

Exercise 3 - Query 2


,season,event_share,n
0,Fall,0.131378,13640
1,Spring,0.071292,14125
2,Summer,0.095820,15456
3,Winter,0.103244,13008


In [108]:
# Final: append Exercise 3 queries back to queries.sql

with open("sql/queries.sql", "a", encoding="utf-8") as f:
    f.write("\n\n-- Exercise 3 Query 1: monthly ranking with QUALIFY\n")
    f.write(ex_sql.strip() + "\n")

    f.write("\n-- Exercise 3 Query 2: event share by season using CTE\n")
    f.write(sql_ex3_q2.strip() + "\n")

print("Exercise 3 SQL appended successfully")

Exercise 3 SQL appended successfully


In [66]:
# Save Exercise 3 results and SQL queries

exercise3_text = """

## Exercise 3 - Additional SQL queries

### Query 1 - QUALIFY
The monthly ranking query shows that JPL ranks above Caltech in the displayed
months. For example, in January the mean hourly charging load is about
21.85 kWh at JPL versus 8.55 kWh at Caltech.

### Query 2 - CTE
Event share by season:
- Fall: 0.131378
- Spring: 0.071292
- Summer: 0.095820
- Winter: 0.103244

Conclusion: High-load events are most frequent in Fall and least frequent
in Spring in this dataset.
"""

with open("report/exercises_C.md", "a", encoding="utf-8") as f:
    f.write(exercise3_text)


# Append the two Exercise 3 SQL queries to queries.sql
with open("sql/queries.sql", "a", encoding="utf-8") as f:
    f.write("\n\n-- Exercise 3 Query 1: monthly ranking with QUALIFY\n")
    f.write(ex_sql.strip() + "\n")

    f.write("\n-- Exercise 3 Query 2: event share by season using CTE\n")
    f.write(sql_ex3_q2.strip() + "\n")

print("saved Exercise 3 to report/exercises_C.md")
print("appended Exercise 3 queries to sql/queries.sql")

saved Exercise 3 to report/exercises_C.md
appended Exercise 3 queries to sql/queries.sql


In [109]:
# Final Notebook C checklist

import os
import pandas as pd

required_files = [
    "data/raw/primary_snapshot.csv",
    "data/processed/clean.csv",
    "data/processed/feat.csv",
    "data/processed/feat.parquet",
    "data/processed/manifest.json",
    "report/data_sources.csv",
    "report/table_source_comparison.csv",
    "report/table_cleaning_log.csv",
    "report/eda_notes.md",
    "report/table_q1.csv",
    "report/table_q2.csv",
    "report/table_q3.csv",
    "report/table_q4.csv",
    "report/table_rq1_tests.csv",
    "report/exercises_C.md",
    "report/ai_audit_log.csv",
    "sql/queries.sql",
    "sql/features.sql"
]

print("=== REQUIRED FILES ===")
all_ok = True

for f in required_files:
    ok = os.path.exists(f)
    print("OK " if ok else "MISSING ", f)
    all_ok &= ok

print("\n=== DATA CHECK ===")

print("feat rows:", len(feat))
print("feat columns:", feat.shape[1])
print("units:", feat[UNIT_COL].nunique())
print("duplicates:", feat[[UNIT_COL, TIME_COL]].duplicated().sum())
print("missing target:", feat[TARGET_COL].isna().sum())

audit_df = pd.read_csv("report/ai_audit_log.csv")

print("\n=== AI AUDIT ===")
print("audit entries:", len(audit_df))
print(
    "hallucination entries:",
    audit_df["hallucination"].astype(bool).sum()
)

print("\nFINAL STATUS:", "OK" if all_ok else "CHECK MISSING FILES")

=== REQUIRED FILES ===
OK  data/raw/primary_snapshot.csv
OK  data/processed/clean.csv
OK  data/processed/feat.csv
OK  data/processed/feat.parquet
OK  data/processed/manifest.json
OK  report/data_sources.csv
OK  report/table_source_comparison.csv
OK  report/table_cleaning_log.csv
OK  report/eda_notes.md
OK  report/table_q1.csv
OK  report/table_q2.csv
OK  report/table_q3.csv
OK  report/table_q4.csv
OK  report/table_rq1_tests.csv
OK  report/exercises_C.md
OK  report/ai_audit_log.csv
OK  sql/queries.sql
OK  sql/features.sql

=== DATA CHECK ===
feat rows: 56229
feat columns: 26
units: 2
duplicates: 0
missing target: 0

=== AI AUDIT ===
audit entries: 6
hallucination entries: 1

FINAL STATUS: OK


### Tự đánh giá Notebook_C (điền trước khi gửi giảng viên)

| Tiêu chí | Tự chấm (0-2) | Bằng chứng |
|---|---|---|
| Nguồn và giấy phép rõ ràng | | report/data_sources.csv |
| So sánh hai nguồn có số | | table_source_comparison.csv |
| Làm sạch có nhật ký | | table_cleaning_log.csv |
| EDA có nhận xét kèm số | | eda_notes.md |
| SQL không rò rỉ, có kiểm tra | | features.sql, ô kiểm tra rò rỉ |
| Bài tập mở rộng | | exercises_C.md |
| AI Audit Log | | ai_audit_log.csv |

## Lỗi thường gặp và cách sửa

| Lỗi | Nguyên nhân | Cách sửa |
|---|---|---|
| `KeyError: 'ts'` | tên cột trong file khác CONFIG | in `df.columns`, sửa `TIME_COL` |
| Rò rỉ tương lai (điểm quá tốt) | dùng `shift(-k)` hoặc trung bình cả chuỗi khi tạo đặc trưng | chỉ dùng `LAG` và cửa sổ `PRECEDING`; kiểm tra `feat.ts` của đặc trưng luôn nhỏ hơn mục tiêu |
| Trùng dòng theo đơn vị và thời gian | nguồn có hai bản ghi cùng thời điểm | `drop_duplicates([UNIT_COL, TIME_COL])` rồi ghi vào nhật ký làm sạch |
| Lưới thời gian thiếu bước | không `resample(FREQ)` | resample và đếm bước thiếu theo đơn vị |
| Ghép nguồn phụ mất nửa dòng | lệch múi giờ hoặc lệch tần suất | đưa cả hai về UTC hoặc giờ địa phương thống nhất, dùng `merge_asof` với `tolerance` |
| DuckDB `Binder Error` | tên cột là từ khoá (`do`, `at`, `year`) hoặc trùng tên bảng | đổi tên cột, dùng dấu ngoặc kép |
| `MemoryError` | nạp toàn bộ file lớn bằng pandas | dùng `duckdb.read_csv_auto` và lọc sớm |
| Chronos không cài được | thiếu `pip install chronos-forecasting` | bỏ qua ô mô hình nền, ghi vào hạn chế |

## Checklist Notebook_C trước khi bàn giao (tick từng dòng)

- [ ] Ba file CSV: `data/raw/primary_snapshot.csv`, `data/processed/clean.csv`, `data/processed/feat.csv` mở được bằng Excel, số dòng khớp manifest
- [ ] `report/data_sources.csv` có đủ hai nguồn với URL, giấy phép, số dòng, khoảng thời gian
- [ ] `report/table_source_comparison.csv` và một câu nhận xét về mức nhất quán giữa hai nguồn
- [ ] `report/table_cleaning_log.csv` giải thích mọi dòng bị bỏ
- [ ] 9 hình EDA có caption nêu số liệu; `report/eda_notes.md` có 5 nhận xét kèm số
- [ ] `sql/queries.sql` 4 truy vấn RQ1 chạy qua; `report/table_q1..q4.csv`; `report/table_rq1_tests.csv` có p-value
- [ ] `sql/features.sql` không dùng LEAD ngoài mục tiêu; ô kiểm tra rò rỉ qua
- [ ] Test cases qua; `feat.parquet` và `manifest.json` gửi cho Sinh viên B
- [ ] AI Audit Log có ít nhất 6 mục cho tuần 1 và 2, ít nhất một mục `hallucination=True`

## Mẫu viết phần Data và RQ1 cho bài báo (điền số từ các bảng)

**Data.** We use {PRIMARY_SOURCE} ({license}, {rows} rows, {units} units, {start} to {end}, frequency {FREQ}) merged with {SECOND_SOURCE} ({share}% of rows have covariates). Cleaning removed {removed} rows (Table cleaning log); the two sources agree on {metric} (Table source comparison).

**RQ1.** The target varies most with {calendar or covariate}: seasonal strength {value} by {period} versus {value} by {period} (Figure EDA 4); units differ by a factor of {ratio} (Figure EDA 5); the covariate {name} leads the target by {lag} steps with correlation {rho} (Figure EDA 7); autocorrelation at the seasonal lag is {acf}, so the seasonal naive baseline is expected to be strong (Figure EDA 8). Kruskal-Wallis across units gives p = {p} (Table RQ1 tests).